# Machine Vision 
---
### Prepared by: Berkan

##Central Projection
Visualise the perspective projection (pinhole camera model) of a unit cube centred at the world origin **(0, 0, 0)** onto an image plane situated at a distance of **f = 1** unit. Project the cube for two different camera configurations (Camera A and Camera B) and also show the world coordinate axes in the projected image.

## Solution

1. First, import the required libraries. Only **NumPy** and **Matplotlib** are needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

2. Define the **unit cube** centred at the world origin. It has 8 vertices, each coordinate being ±0.5, and 12 edges connecting adjacent vertices.

In [ ]:
# 8 vertices of the unit cube centred at the world origin
vertices_W = np.array([
    [-0.5, -0.5, -0.5],
    [ 0.5, -0.5, -0.5],
    [ 0.5,  0.5, -0.5],
    [-0.5,  0.5, -0.5],
    [-0.5, -0.5,  0.5],
    [ 0.5, -0.5,  0.5],
    [ 0.5,  0.5,  0.5],
    [-0.5,  0.5,  0.5],
], dtype=float)

# 12 edges as index pairs
edges = [
    (0,1),(1,2),(2,3),(3,0),  # bottom face
    (4,5),(5,6),(6,7),(7,4),  # top face
    (0,4),(1,5),(2,6),(3,7),  # vertical edges
]

3. Define the **camera parameters** for both Camera A and Camera B. The rotation matrix `ᵂRc` describes the orientation of the camera coordinate system in the world, and the translation vector `ᵂtc` gives the camera position in the world.

In [ ]:
# ── Camera A ──────────────────────────────────────────────────
R_C_A = np.array([
    [ 0,  0, -1],
    [ 1,  0,  0],
    [ 0, -1,  0]
], dtype=float)
t_C_A = np.array([3, 0, 0], dtype=float)

# ── Camera B ──────────────────────────────────────────────────
R_C_B = np.array([
    [-0.7071,  0.3536, -0.6124],
    [ 0.7071,  0.3536, -0.6124],
    [ 0.0000, -0.8660, -0.5000]
], dtype=float)
t_C_B = np.array([4, 3, 2], dtype=float)

f = 1.0  # focal length (distance to image plane)

4. Implement the **world-to-camera transformation**. The camera coordinate frame is defined by its position `ᵂtc` and orientation `ᵂRc` (columns = camera X, Y, Z axes in world). To express world points in the camera frame:

$$\mathbf{p}_C = {}^W R_C^\top \cdot (\mathbf{p}_W - {}^W t_C)$$

The transpose of `ᵂRc` is used because rotation matrices are orthogonal (inverse = transpose), effectively rotating back from world to camera orientation.

In [ ]:
def world_to_camera(points_W, R_C, t_C):
    """
    Transform world-frame points into camera-frame coordinates.
    p_C = R_C^T * (p_W - t_C)
    """
    R_CW = R_C.T          # world-to-camera rotation (transpose = inverse for orthogonal matrix)
    return (points_W - t_C) @ R_CW.T

# Quick sanity check: cube vertices in Camera A frame (Z must be > 0 to be in front)
verts_C_A = world_to_camera(vertices_W, R_C_A, t_C_A)
print('Camera A – Z range of cube vertices:', verts_C_A[:, 2].min(), 'to', verts_C_A[:, 2].max())
verts_C_B = world_to_camera(vertices_W, R_C_B, t_C_B)
print('Camera B – Z range of cube vertices:', verts_C_B[:, 2].min(), 'to', verts_C_B[:, 2].max())

5. Implement the **pinhole (central) projection**. All rays from 3D scene points pass through the single camera centre and intersect the image plane at distance *f*. The projection formula is:

$$x_{img} = f \cdot \frac{X_C}{Z_C}, \quad y_{img} = f \cdot \frac{Y_C}{Z_C}$$

Points with $Z_C \leq 0$ lie behind the camera and are discarded.

In [ ]:
def pinhole_project(points_C, f=1.0):
    """
    Project 3D camera-frame points onto the image plane.
    Returns 2D image coordinates and a validity mask (Z > 0).
    """
    valid = points_C[:, 2] > 0
    x_img = np.where(valid, f * points_C[:, 0] / points_C[:, 2], np.nan)
    y_img = np.where(valid, f * points_C[:, 1] / points_C[:, 2], np.nan)
    return np.stack([x_img, y_img], axis=1), valid

6. Define a helper that projects the **world coordinate axes** (X, Y, Z) into the image so they can be drawn as coloured arrows. The origin **(0,0,0)** and the unit-length tips of each axis are projected through the same pipeline.

In [ ]:
def project_axes(R_C, t_C, f=1.0, length=0.7):
    """
    Project world coordinate axes into image coordinates.
    Returns dict with 'origin' and 'tip' 2D points for X, Y, Z.
    """
    pts_W = np.array([
        [0, 0, 0],          # origin
        [length, 0, 0],     # X tip
        [0, length, 0],     # Y tip
        [0, 0, length],     # Z tip
    ], dtype=float)
    pts_C  = world_to_camera(pts_W, R_C, t_C)
    pts_2D, valid = pinhole_project(pts_C, f)
    origin = pts_2D[0]
    return {
        'X': {'origin': origin, 'tip': pts_2D[1], 'valid': valid[0] and valid[1]},
        'Y': {'origin': origin, 'tip': pts_2D[2], 'valid': valid[0] and valid[2]},
        'Z': {'origin': origin, 'tip': pts_2D[3], 'valid': valid[0] and valid[3]},
    }

7. Visualise the results. For each camera, two panels are shown side by side:
- **Left (3D scene):** the world with the cube, the world coordinate axes, and the camera position/orientation.
- **Right (2D projection):** the projected image with cube edges and projected world axes (red = X, green = Y, blue = Z).

The perspective effect is clearly visible in the 2D images — parallel cube edges converge toward vanishing points, which is the characteristic behaviour of the pinhole camera model.

In [ ]:
AXIS_COLORS = {'X': 'red', 'Y': 'green', 'Z': 'blue'}

def draw_3d_scene(ax, R_C, t_C, label):
    """Draw 3D world scene with cube, world axes, and camera."""
    # Cube edges
    for i, j in edges:
        v1, v2 = vertices_W[i], vertices_W[j]
        ax.plot([v1[0],v2[0]], [v1[1],v2[1]], [v1[2],v2[2]], 'w-', lw=1.5, alpha=0.8)

    # World axes
    L = 0.8
    for vec, col, lbl in [([L,0,0],'red','X'), ([0,L,0],'green','Y'), ([0,0,L],'blue','Z')]:
        ax.quiver(0,0,0,*vec, color=col, lw=2, arrow_length_ratio=0.25)
        ax.text(vec[0]+0.05, vec[1]+0.05, vec[2]+0.05, lbl, color=col, fontsize=9, fontweight='bold')

    # Camera position
    ax.scatter(*t_C, color='yellow', s=60, zorder=5)
    ax.text(t_C[0]+0.1, t_C[1]+0.1, t_C[2]+0.1, f'Cam {label}', color='yellow', fontsize=9, fontweight='bold')

    # Camera orientation axes (dashed)
    CL = 0.6
    for k, col in enumerate(['#ff8888','#88ff88','#8888ff']):
        d = R_C[:, k] * CL
        ax.quiver(*t_C, *d, color=col, lw=1.5, arrow_length_ratio=0.3, linestyle='dashed')

    # --- YENİ EKLENEN: 3D Eksenleri Eşit Ölçekleme (Equal Aspect Ratio) ---
    x_limits = ax.get_xlim3d()
    y_limits = ax.get_ylim3d()
    z_limits = ax.get_zlim3d()

    x_range = abs(x_limits[1] - x_limits[0])
    x_middle = np.mean(x_limits)
    y_range = abs(y_limits[1] - y_limits[0])
    y_middle = np.mean(y_limits)
    z_range = abs(z_limits[1] - z_limits[0])
    z_middle = np.mean(z_limits)

    plot_radius = 0.5 * max([x_range, y_range, z_range])

    ax.set_xlim3d([x_middle - plot_radius, x_middle + plot_radius])
    ax.set_ylim3d([y_middle - plot_radius, y_middle + plot_radius])
    ax.set_zlim3d([z_middle - plot_radius, z_middle + plot_radius])
    ax.set_box_aspect([1, 1, 1])
    # ----------------------------------------------------------------------

    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(f'3D Scene — Camera {label}', fontweight='bold')

def draw_projection(ax, R_C, t_C, label, f=1.0):
    """Draw 2D projected image of the cube and world axes."""
    # Project cube vertices
    verts_C, valid = pinhole_project(world_to_camera(vertices_W, R_C, t_C), f)

    # Draw cube edges
    for i, j in edges:
        if valid[i] and valid[j]:
            p1, p2 = verts_C[i], verts_C[j]
            ax.plot([p1[0],p2[0]], [p1[1],p2[1]], 'w-', lw=2)

    # Draw projected vertices
    ax.scatter(verts_C[valid,0], verts_C[valid,1], color='white', s=20, zorder=5)

    # Draw world coordinate axes
    axes_2D = project_axes(R_C, t_C, f, length=0.5)
    for name, info in axes_2D.items():
        if info['valid']:
            ox, oy = info['origin']
            tx, ty = info['tip']
            ax.annotate('', xy=(tx,ty), xytext=(ox,oy),
                        arrowprops=dict(arrowstyle='->', color=AXIS_COLORS[name], lw=2.5, mutation_scale=16))
            ax.text(tx+0.005, ty+0.005, name, color=AXIS_COLORS[name], fontsize=11, fontweight='bold')

    ax.set_aspect('equal')
    
    #(Upside Down Fix) ---
    ax.invert_yaxis() 
    # ----------------------------------------------------------

    ax.set_facecolor('#111111')
    ax.tick_params(colors='gray')
    ax.set_xlabel('x (image)'); ax.set_ylabel('y (image)')
    ax.set_title(f'Projected Image — Camera {label}  (f={f})', fontweight='bold')
    ax.grid(True, color='#222222', lw=0.5)


# ── Draw everything ───────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 10))
fig.patch.set_facecolor('#1a1a1a')

ax3A = fig.add_subplot(2, 2, 1, projection='3d')
ax3B = fig.add_subplot(2, 2, 2, projection='3d')
ax2A = fig.add_subplot(2, 2, 3)
ax2B = fig.add_subplot(2, 2, 4)

for ax in [ax3A, ax3B]:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='gray', labelsize=7)
    ax.xaxis.pane.set_edgecolor('#333'); ax.yaxis.pane.set_edgecolor('#333'); ax.zaxis.pane.set_edgecolor('#333')
    ax.xaxis.pane.fill = ax.yaxis.pane.fill = ax.zaxis.pane.fill = False

draw_3d_scene(ax3A, R_C_A, t_C_A, 'A')
draw_3d_scene(ax3B, R_C_B, t_C_B, 'B')
draw_projection(ax2A, R_C_A, t_C_A, 'A', f)
draw_projection(ax2B, R_C_B, t_C_B, 'B', f)

for ax in [ax3A, ax3B, ax2A, ax2B]:
    ax.title.set_color('white')
    ax.xaxis.label.set_color('gray'); ax.yaxis.label.set_color('gray')

fig.suptitle('Assignment 08 – Central Projection (Pinhole Camera Model)',
             color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('central_projection.png', dpi=150, bbox_inches='tight', facecolor='#1a1a1a')
plt.show()

8. **Summary of results:**

The implementation correctly applies the pinhole camera model for both camera configurations:

- **Camera A** (`ᵂtc = [3,0,0]`): The camera looks along the world -X axis. The cube appears nearly square (frontal view) since the camera is aligned with the cube face. The world Y and X axes appear short due to the viewing angle, while Z points toward the viewer.

- **Camera B** (`ᵂtc = [4,3,2]`): The camera is positioned diagonally above and to the side. This produces a clear perspective view of the cube showing three faces simultaneously — the classic isometric-like perspective appearance. All three world axes are clearly visible.

In both cases, the parallel edges of the cube converge toward vanishing points in the projected image, which is the defining characteristic of **central (perspective) projection** — as opposed to orthographic projection where parallel lines remain parallel.